# ConstructionBots Experiment Lab

**ConstructionBots** simulates a robot "factory": a team of small mobile robots builds a LEGO model by fetching
every part and sub-assembly, carrying it (alone or in teams) to the right place, and putting it in position.
The central question is **who should carry what, and in which order**: the *task assignment*. This notebook lets
you try ten assignment algorithms (heuristic rules, exact optimization and metaheuristics), see what they
plan, watch the robots carry it out, and compare the algorithms fairly.

You do not need to write code. Change values only in the cells marked **✏️ EDIT ME**, then run the cells from top to bottom.

### Quick start (about 5 minutes)

1. Run the **Setup** cell in Section 2. Every line of its table should say **OK**.
2. Run Sections 3 and 4 unchanged. They build the Tractor with the **Genetic algorithm** and show the result,
   the robots' planned schedule, how the search improved it, and a 3-D animation.
3. Run Section 5, the **algorithm benchmark**: nine algorithms on the same problem, with comparison charts.
4. Now experiment: change `ASSIGNMENT_METHOD` in Section 3 or the list `ALGORITHMS_TO_COMPARE` in Section 5 and run again.

> **Version note.** Written for and tested with ConstructionBots 0.1.5 (this fork, branch `fork-work`), Julia 1.10.12,
> on Windows 11. Every run records the exact code version it used. See `notebooks/NOTEBOOK_USER_GUIDE.md`
> for a longer guide and `notebooks/SETUP.md` for installation.

## Contents

1. [Welcome: what is simulated](#1.-Welcome:-what-is-simulated)
2. [Setup and environment check](#2.-Setup-and-environment-check)
3. [Experiment settings (the cell you edit)](#3.-Experiment-settings)
4. [Run one experiment and see it](#4.-Run-one-experiment-and-see-it)
5. [Compare algorithms (benchmark)](#5.-Compare-algorithms-(benchmark))
6. [Repeated trials (optional)](#6.-Repeated-trials-(optional))
7. [Results files and CSV export](#7.-Results-files-and-CSV-export)
8. [Troubleshooting](#8.-Troubleshooting)
9. [Licences: what is free and what needs a licence](#9.-Licences)
10. [Known limitations](#10.-Known-limitations)

## 1. Welcome: what is simulated

**What the simulator models**

| Question | Handled by |
|---|---|
| How many robots must carry each part, and how they stand around it | transport-team configuration |
| Where each sub-assembly is built and where parts are delivered | construction-site layout ("staging plan") |
| Which robot fetches which part, in which order | **task assignment** (you choose the method) |
| How robots drive without crashing into each other or into building sites | **movement / collision avoidance** (you choose the profile) |

**What is abstracted away:** the physics of gripping and snapping LEGO pieces together, and detailed robot dynamics.
Robots are simple cylinders; parts are lifted into place by the simulator.

**What an experiment is:** one simulation of one model with one set of choices. It produces
**statistics** (most importantly the *completion time*, also called *makespan*, and a *safety record*: how close
robots came to each other) and, optionally, a 3-D **animation** you can open in a web browser.

**What you can change:** the model, the number of robots, the task-assignment algorithm (and its solver or
search budget), the movement profile, the random seed, and a time limit. Everything else stays at the repository's defaults.

### How to use this notebook

* Run a cell by clicking it and pressing **Shift + Enter**. Run cells **from top to bottom**.
* A cell is running while it shows `[*]` (Jupyter) or a spinning icon (VS Code).
* **Start fresh at any time:** use **Restart** (VS Code: *Restart* button at the top; Jupyter: *Kernel → Restart*),
  then run the cells again from the top. Nothing on disk is lost; every run is saved in its own new folder.
* The **first** experiment after a restart is slow (often 1–3 minutes) because Julia compiles the code. Later runs are fast.
* You only need to edit the cells marked **✏️ EDIT ME**: the experiment settings in Section 3 and, for a
  comparison, the comparison settings in Section 5.

## 2. Setup and environment check

Before the first use on a computer, the software must be installed once. That is described step by step in
**`notebooks/SETUP.md`** (Julia, the LDraw parts library, the Python `rvo2` module, and the Jupyter kernel).

The next cell **does not install anything**. It:

1. finds the ConstructionBots folder on this computer and activates its Julia project,
2. loads the simulator and this notebook's helper code (`notebooks/notebook_helpers.jl`),
3. checks every requirement and prints a table. Each line should say **OK**. A **WARNING** on Gurobi is normal
   if you have no Gurobi licence.

If the cell cannot find the repository, fill in `REPO_ROOT_OVERRIDE` with the folder path (keep the `raw"..."`).
If your LDraw parts library is not in `Documents\ldraw`, fill in `LDRAW_LIBRARY_OVERRIDE`.

In [1]:
# ======================= SETUP (run once per session) =======================
# Normally leave both as `nothing`. Example of a manual path:
#   REPO_ROOT_OVERRIDE = raw"C:\Users\yourname\Documents\GitHub\ConstructionBots.jl"
REPO_ROOT_OVERRIDE = nothing
LDRAW_LIBRARY_OVERRIDE = nothing
# ============================================================================

import Pkg, TOML

function find_constructionbots_root(override)
    is_repo(d) = isfile(joinpath(d, "Project.toml")) &&
                 get(TOML.parsefile(joinpath(d, "Project.toml")), "name", "") == "ConstructionBots" &&
                 isfile(joinpath(d, "notebooks", "notebook_helpers.jl"))
    if override !== nothing
        is_repo(override) && return abspath(override)
        error("REPO_ROOT_OVERRIDE = $(repr(override)) is not a ConstructionBots.jl folder " *
              "(it must contain Project.toml and notebooks/notebook_helpers.jl).")
    end
    starts = String[pwd()]
    haskey(ENV, "CONSTRUCTIONBOTS_ROOT") && pushfirst!(starts, ENV["CONSTRUCTIONBOTS_ROOT"])
    Base.active_project() !== nothing && push!(starts, dirname(Base.active_project()))
    for start in starts
        d = abspath(start)
        while true
            is_repo(d) && return d
            parent = dirname(d)
            parent == d && break
            d = parent
        end
    end
    error("Could not find the ConstructionBots.jl folder automatically. Set REPO_ROOT_OVERRIDE " *
          "at the top of this cell to its full path, then run the cell again.")
end

REPO_ROOT = find_constructionbots_root(REPO_ROOT_OVERRIDE)
println("Repository folder: ", REPO_ROOT)
Pkg.activate(REPO_ROOT)

if !isdefined(Main, :ExperimentLab)   # load once per session; restart the kernel to reload
    println("Loading ConstructionBots and the notebook helpers (can take a minute)...")
    include(joinpath(REPO_ROOT, "notebooks", "notebook_helpers.jl"))
end
using .ExperimentLab
lab_setup(; repo_root=REPO_ROOT, ldraw_library=LDRAW_LIBRARY_OVERRIDE)

ENVIRONMENT_OK = check_environment()
if ENVIRONMENT_OK
    println("\nEverything required is ready. Continue with Section 3.")
else
    println("\nSome required item shows PROBLEM above. Follow the hint in its 'Details' column ",
            "and notebooks/SETUP.md, then restart the kernel and run this cell again.")
end

Repository folder: c:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl


  Activating project at `c:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl`


Loading ConstructionBots and the notebook helpers (can take a minute)...


Check,Status,Details
Julia,OK,version 1.10.12
Active Julia project,OK,c:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl\Project.toml
ConstructionBots package,OK,version 0.1.5 loaded from c:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl\src
Code version,OK,"source hash 1ce4205b64cd, commit 6de2c49"
Model files (LDraw_files/),OK,all 15 registered models found
LDraw Parts Library,OK,C:\Users\Ali Akarma\Documents\ldraw (24856 part files)
Python used by PyCall,OK,C:\Conda-Envs\lego_rvo2\python.exe (Python 3.7.16)
Python module rvo2,OK,rvo2 found at C:\Conda-Envs\lego_rvo2\lib\site-packages\rvo2.pyd
Solver: HiGHS,OK,HiGHS 1.26.0 is ready.
Solver: GLPK,OK,GLPK.jl 1.2.1 is ready.



Everything required is ready. Continue with Section 3.


### What can I choose?

Run the next cell to list every model, assignment method, solver and movement profile with a short
explanation. Use the names **exactly as shown** (capital letters do not matter).

In [2]:
show_models()
show_options()

MODEL,Size,Parts,Assemblies,Default robots,Reference time*,
Tractor,small,20,8,10,about 2 s,
Colored 8x8,small,33,1,24,about 4 s,
TIE Fighter (mini),small,44,4,15,about 7 s,
X-Wing (mini),small,61,12,20,about 9 s,
Imperial Shuttle (mini),medium,84,5,20,about 13 s,
Quad Nested,medium,85,21,50,about 26 s,
AT-TE Walker (mini),medium,100,22,35,about 23 s,
X-Wing + TIE (mini),medium,105,17,20,about 26 s,
X-Wing,large,309,28,50,about 3 min,needs ALLOW_LARGE_MODELS
Passenger Plane,large,326,28,50,about 4 min,needs ALLOW_LARGE_MODELS


ASSIGNMENT_METHOD,Family,Uses a solver?,What it does
Greedy,Heuristic rule,no,Rule: the task that a robot team can finish earliest goes first (the repository's GreedyOrderedAssignment). Fast; the recommended baseline.
Greedy with travel,Heuristic rule,no,"Rule: Greedy, except that each robot's finish time includes its earlier travel. This is the decoder the metaheuristics use, so it shows what they gain from the decoder alone."
Most complex first,Heuristic rule,no,Rule: the task that needs the largest robot team (then has the most LEGO parts) goes first.
Nearest station first,Heuristic rule,no,"Rule: the task whose pick-up station the free robots reach soonest goes first (travel time only, ignoring when the robots become free)."
MILP,Exact optimization,yes,"Mixed-integer linear program solved by an optimization solver, stopped at the time limit."
Greedy + MILP warm start,Exact optimization,yes,Runs Greedy first and gives its answer to the MILP solver as a starting point.
Genetic algorithm,Metaheuristic,no,"Metaheuristic: evolves a population of task-priority orders (selection, crossover, mutation)."
Particle swarm,Metaheuristic,no,Metaheuristic: a swarm of task-priority orders moves towards the best ones found (PSO).
Simulated annealing,Metaheuristic,no,"Metaheuristic: changes one task-priority order step by step, sometimes accepting a worse one early on."
Random search,Metaheuristic,no,"Control for the metaheuristics: the same budget of candidates, drawn at random with no search operators."


SOLVER,Cost,Notes
HiGHS,free,Open-source (MIT licence). Recommended default.
GLPK,free,Open-source (GPL-3.0). Cannot be used with 'Greedy + MILP warm start'.
Gurobi,licence required,Commercial. Needs your own valid Gurobi licence (free academic licences exist).


MOVEMENT_PROFILE,RVO2,Tangent Bug,Dispersion,Notes
Recommended,on,on,on,RVO2 + Tangent Bug + Dispersion all on (the configuration of the repository's hosted demos).
No RVO2,off,on,on,"Turns off RVO2 reciprocal collision avoidance. Note: in this version of the code the dispersion velocity adjustment is only applied while RVO2 is on, so dispersion has little effect here."
No Tangent Bug,on,off,on,Robots head straight for their goal instead of steering around construction sites.
No Dispersion,on,on,off,Ablation only: the simulator itself warns that RVO2 on + dispersion off is not recommended.
No avoidance methods,off,off,off,All three modules off: robots move straight to their goals and may pass through each other.


## 3. Experiment settings

### The algorithms you can compare

All ten algorithms solve the same problem: they decide **which robots carry which part or sub-assembly, and
in which order**. All of them respect the model's build order and the size of every transport team, and the same
movement simulation runs afterwards. So any difference in the results comes from the assignment, and from how its
plan happens to play out in traffic.

| Family | `ASSIGNMENT_METHOD` | How it decides | Computer time (Tractor) |
|---|---|---|---|
| **Heuristic rules** (fixed rules) | `"Greedy"` | Again and again, pick the open task that a robot team can **finish earliest**, and send the robots that get there first. The repository's own method and the baseline. | well under 1 s |
| | `"Greedy with travel"` | The same as Greedy, except that each robot's finish time includes its earlier driving. This is the decoder the metaheuristics use, so it shows what they gain from the decoder alone. | well under 1 s |
| | `"Most complex first"` | Pick the open task that needs the **largest robot team** (then: whose cargo has the most LEGO parts). Big jobs first, so they do not hold up the end. | well under 1 s |
| | `"Nearest station first"` | Pick the open task whose pick-up station the free robots can **reach soonest**, judged by driving time only (not by when the robots become free). | well under 1 s |
| **Exact optimization** (MILP) | `"MILP"` | Writes the whole assignment as a **mixed-integer linear program** and lets a solver (HiGHS by default) search for the schedule with the shortest planned completion time. Stops at the time limit with the best schedule found so far, which is usually **not** the optimum: read a MILP result as "this solver for this long". | up to the time limit |
| | `"Greedy + MILP warm start"` | The same, but the solver starts from Greedy's schedule. | up to the time limit |
| **Metaheuristics** | `"Genetic algorithm"` (`"GA"`) | Keeps a **population** of candidate task orders. The better ones are combined (crossover) and slightly changed (mutation), generation after generation. | a few seconds |
| | `"Particle swarm"` (`"PSO"`) | A **swarm** of candidate task orders. Each one moves towards its own best order and the swarm's best order. | a few seconds |
| | `"Simulated annealing"` (`"SA"`) | **One** candidate order, changed a little at a time. Early on, a worse change is sometimes accepted to escape dead ends; later almost never ("cooling"). | a few seconds |
| | `"Random search"` | The **control**: the same budget of candidates, drawn at random, with no search operators. If a metaheuristic does not beat it, its operators add nothing. | a few seconds |

**How the metaheuristics work here.** A candidate solution is a list of numbers, one per transport task (so-called
*random keys*): a higher number means "assign this task earlier". A candidate is turned into a full schedule with the
same rules as Greedy (build order respected, the nearest free robots matched to the task) and is scored by its
**planned completion time**, including the robots' driving. The search starts from Greedy's task order and keeps the
best schedule it finds. Greedy's own schedule counts as a candidate too, so the **plan** is never worse than Greedy's.
A better plan does **not** guarantee a faster simulation, because the plan ignores traffic: judge the algorithms by the
simulated completion time over several seeds (Section 6). On one model tested (Colored 8×8), plain metaheuristics planned
15 % shorter than Greedy and took twice as long in simulation; `SIMULATE_TOP_PLANS` in the Advanced settings guards against
this. Measured results for six models are in `docs/EXPERIMENTS.md`. Each algorithm gets the same budget: `SEARCH_ITERATIONS × SEARCH_POPULATION` schedules
(default 25 × 16 = 400, a few seconds on the Tractor). It stops early at `TIME_LIMIT_SECONDS`. The random numbers
come from `RANDOM_SEED`, so the same settings give the same search.

### The settings

Change **one kind** of setting at a time. The *assignment method* decides **who does what**; the *movement profile*
decides **how robots drive**. If you change both and the result changes, you cannot tell which change caused it.

| Setting | What it means | When to change it |
|---|---|---|
| `MODEL` | Which LEGO model is built. Start with `"Tractor"` (20 parts, seconds). | To test a larger problem. |
| `NUMBER_OF_ROBOTS` | `nothing` = the model's recommended number (Tractor: 10), or a whole number. | To study team size. Too few robots can make the assignment impossible (the run then reports *failed*). |
| `ASSIGNMENT_METHOD` | One of the ten algorithms above (capital letters do not matter). | To compare algorithms. |
| `TIME_LIMIT_SECONDS` | The longest time the MILP solver or a metaheuristic may search per run. | Larger values may give better MILP results but take longer. Not used by the rules. |
| `SOLVER` | MILP methods only: `"HiGHS"` (free, recommended), `"GLPK"` (free), `"Gurobi"` (needs a licence). | To compare solvers. GLPK cannot be used with the warm start. |
| `SEARCH_ITERATIONS`, `SEARCH_POPULATION` | Metaheuristics only: how many iterations, and how many candidate schedules per iteration. | Larger values search longer and often find better plans. |
| `MOVEMENT_PROFILE` | `"Recommended"` (all avoidance methods on), `"No RVO2"`, `"No Tangent Bug"`, `"No Dispersion"`, `"No avoidance methods"`, or `"Custom"` (Advanced settings). | To measure what each movement method contributes. |
| `RANDOM_SEED` | A whole number that fixes the robots' starting positions (and the metaheuristics' random choices). The parts' locations stay the same. | Use different seeds to repeat an experiment fairly (Section 6). |
| `SAVE_ANIMATION` | `true` records a 3-D animation of the run, shown in Section 4. Adds a few seconds for small models. | `false` for faster runs. |
| `SAVE_RESULTS` | `true` keeps a folder with the statistics, settings and log of this run. | Keep `true` for real experiments. |

**Glossary.** *Makespan* = completion time. *MILP* = mixed-integer linear program. *RVO2* = "reciprocal velocity
obstacles": robots adjust their speed to avoid each other. *Tangent Bug* = robots steer around construction sites
in their way. *Dispersion* = a "repulsion field" that spreads robots out around busy areas.

In [25]:
# =====================================================================
#  ✏️ EDIT ME: EXPERIMENT SETTINGS (normally the only cell you change)
# =====================================================================
MODEL                = "Tractor"             # see the "Available models" table; start small
NUMBER_OF_ROBOTS     = nothing               # nothing = model default (Tractor: 10), or e.g. 12

ASSIGNMENT_METHOD    = "Greedy"   # who carries what. Choose one:
#   heuristic rules : "Greedy", "Greedy with travel", "Most complex first", "Nearest station first"
#   exact (MILP)    : "MILP", "Greedy + MILP warm start"
#   metaheuristics  : "Genetic algorithm", "Particle swarm", "Simulated annealing", "Random search"   (or "GA", "PSO", "SA")

TIME_LIMIT_SECONDS   = 60                    # the most time the MILP solver or a metaheuristic may search
SOLVER               = "Gurobi"               # MILP only: "HiGHS" (free), "GLPK" (free), "Gurobi" (licence)
SEARCH_ITERATIONS    = 25                    # metaheuristics only: iterations (generations)
SEARCH_POPULATION    = 16                    # metaheuristics only: candidate schedules per iteration

MOVEMENT_PROFILE     = "Recommended"         # "Recommended", "No RVO2", "No Tangent Bug",
                                             # "No Dispersion", "No avoidance methods", "Custom"
RANDOM_SEED          = 1                     # whole number
SAVE_ANIMATION       = true                  # true = record a 3-D animation (shown in Section 4)
SAVE_RESULTS         = true                  # true = keep a results folder for this run

println("Settings stored. Now run the next cells.")

Settings stored. Now run the next cells.


### Advanced settings (optional)

Most users never change these. They are listed here so nothing is hidden.

| Setting | Meaning |
|---|---|
| `CUSTOM_RVO2`, `CUSTOM_TANGENT_BUG`, `CUSTOM_DISPERSION` | Individual on/off switches, used **only** when `MOVEMENT_PROFILE = "Custom"`. |
| `SMOOTH_ANIMATION` | `true` records the animation at every simulation step (smooth, but much slower). With `false`, robots appear to jump in straight lines between events and may *look* as if they collide. |
| `ALLOW_LARGE_MODELS` | Large models (X-Wing, Passenger Plane, Star Destroyer, Kings Castle, AT-AT, Heavily Nested, Saturn V) can take from minutes to hours. They are refused unless this is `true`. |
| `MAX_SIMULATION_STEPS` | Safety limit on simulation length (simulator default 100000 steps of 1/40 s). |
| `MAX_STEPS_WITHOUT_PROGRESS` | The simulation stops if no task finishes for this many steps (simulator default 10000). Such a run is reported as *incomplete*. |
| `REUSE_PREVIOUS_RESULTS` | `true` (default): if a run with identical settings and the same code version already exists, its saved result is shown at once and nothing is simulated again (the record's status detail says so, and its computer time is 0). `false` simulates every run again, for example when you measure computer time. |
| `SIMULATE_TOP_PLANS` | Metaheuristics only. `0` (default) uses the best plan without simulating the others. Above 0, the run simulates Greedy's plan and this many of the search's best distinct plans, and keeps the fastest **in simulation**, so the simulated result is never worse than Greedy's. Costs that many extra simulations. |
| `SHOW_SIMULATOR_LOG` | `false` saves the simulator's long technical output (including the solver log) to `simulator_log.txt` in the run folder; `true` prints it here. |

In [26]:
# ----------------------- ADVANCED SETTINGS (optional) -----------------------
CUSTOM_RVO2                = true     # used only when MOVEMENT_PROFILE = "Custom"
CUSTOM_TANGENT_BUG         = true     # used only when MOVEMENT_PROFILE = "Custom"
CUSTOM_DISPERSION          = true     # used only when MOVEMENT_PROFILE = "Custom"
SMOOTH_ANIMATION           = false
ALLOW_LARGE_MODELS         = true
MAX_SIMULATION_STEPS       = 100000   # simulator default
MAX_STEPS_WITHOUT_PROGRESS = 10000    # simulator default
REUSE_PREVIOUS_RESULTS     = true     # true = an identical earlier run is reused instead of simulated again
SIMULATE_TOP_PLANS         = 0        # metaheuristics: choose among Greedy and this many best plans by simulating them
SHOW_SIMULATOR_LOG         = false

println("Advanced settings stored.")

Advanced settings stored.


### Check the settings

The next cell turns your settings into an experiment description and checks it **before** anything runs.
If something is wrong it stops with a message that starts with **"Settings problem"** and lists the valid
choices. Fix the value in the settings cell and run both cells again.

In [27]:
CONFIG = make_config(
    model = MODEL, num_robots = NUMBER_OF_ROBOTS, assignment = ASSIGNMENT_METHOD, solver = SOLVER,
    movement = MOVEMENT_PROFILE, seed = RANDOM_SEED, optimizer_time_limit_s = TIME_LIMIT_SECONDS,
    search_iterations = SEARCH_ITERATIONS, search_population = SEARCH_POPULATION,
    simulate_top_plans = SIMULATE_TOP_PLANS,
    save_animation = SAVE_ANIMATION, save_results = SAVE_RESULTS, reuse_results = REUSE_PREVIOUS_RESULTS,
    custom_rvo = CUSTOM_RVO2, custom_tangent_bug = CUSTOM_TANGENT_BUG, custom_dispersion = CUSTOM_DISPERSION,
    smooth_animation = SMOOTH_ANIMATION, allow_large_models = ALLOW_LARGE_MODELS,
    max_steps = MAX_SIMULATION_STEPS, max_iters_no_progress = MAX_STEPS_WITHOUT_PROGRESS,
)
check_settings(CONFIG);

Setting,Value,Passed to the simulator as
Model,Tractor,"20 parts, file LDraw_files/tractor.mpd, scale 0.008"
Number of robots,10,model default
Assignment method,Greedy,Heuristic rule; run_lego_demo(assignment_mode = :greedy)
Solver,not used,Greedy does not use an optimization solver
Movement profile,Recommended,"rvo_flag = true, tangent_bug_flag = true, dispersion_flag = true"
Random seed,1,rng = MersenneTwister(1); sets the robots' starting positions
Save animation,on,updated at task completions
Save results,on,new folder under results\notebook_lab\runs
Reuse earlier results,on,an earlier run with identical settings and code is returned instead of simulating again


## 4. Run one experiment and see it

Run the next cell. It prints the run's ID, then a progress bar while the robots are simulated. On the Tractor the
rules take a few seconds and the metaheuristics a few seconds more. The **first** run after a restart adds 1–3
minutes of one-time compilation. MILP methods can use the whole time limit.

The previous result is cleared **before** the new run starts, so an old result can never be mistaken for the new
one. If the run fails, its status says **FAILED** with the reason. Nothing is reported as a success unless the
simulated model was actually completed.

In [28]:
LAST_RUN = nothing    # forget the previous result first
LAST_RUN = run_experiment(CONFIG; show_simulator_log = SHOW_SIMULATOR_LOG);

[ Info: Listening on: 127.0.0.1:8700, thread id: 1
┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
└ http://127.0.0.1:8700


Run 20261103-233318-247_tractor_greedy_rovk
  Model: Tractor (10 robots, model default) | Assignment: Greedy | Movement: Recommended | Seed: 1
  First run of this configuration type in this session: expect up to a few minutes of one-time compilation.
  Running. The simulator's detailed console output goes to results\notebook_lab\runs\20261103-233318-247_tractor_greedy_rovk\simulator_log.txt (set SHOW_SIMULATOR_LOG = true to see it here).


Processing file... 100%|██████████████████████████████████████████████████| Time: 0:00:18 ( 0.30  s/it)
Simulating...  45%|███████████████████████                           |  ETA: 0:00:01 ( 9.12 ms/it)
   step_num: 219
   n_closed: 156
4m    n_total: 289


Simulating...  68%|██████████████████████████████████                |  ETA: 0:00:01 (12.24 ms/it)
   step_num: 514
   n_closed: 211
4m    n_total: 289


Simulating...  86%|███████████████████████████████████████████       |  ETA: 0:00:01 (14.63 ms/it)
   step_num: 741
   n_closed: 254
4m    n_total: 289


Simulating... 100%|██████████████████████████████████████████████████| Time: 0:00:03 (14.23 ms/it)
[ Info: Server on 127.0.0.1:8700 closing
[ Info: MeshCat server closed.


  Status: SUCCESS (project completed) after 25.85 s of computer time.
  Animation: 64 moving objects, 24499 keyframes. View it with show_animation(...).


### See the results

The next cell shows everything about the run, in this order:

1. **Result table.** The main number is the **completion time (makespan)**: how long the robots took, in the
   simulated world, to finish the model. **Lower is better.** The table below explains every line.
2. **Planned schedule** (a Gantt chart). One row per robot. A grey bar means the robot is driving to its next task;
   a blue bar means its team is carrying a single part, an orange bar a sub-assembly. This is the plan the
   assignment algorithm produced. Hover over a bar to see the task and its times. Short grey bars and few gaps mean
   a tight plan.
3. **Search progress** (metaheuristics only). The best planned completion time after each iteration. Iteration 0
   is the starting point: Greedy's schedule or Greedy's task order, whichever plans better. The curve shows how the
   *plan* improved, not necessarily the simulated completion time.
4. **3-D animation** of the robots carrying out the plan. Press **Play**, drag the slider to move through time,
   drag in the picture to rotate, scroll to zoom, and press **Reset view** if you lose the model. The table above
   it checks the saved file: it says how many objects (robots, parts, transport teams) actually **move** in it.
   The animation is embedded in the notebook, so it works in Jupyter, JupyterLab and VS Code without a server.

| Measurement | Unit | Meaning |
|---|---|---|
| **Completion time (makespan)** | simulated seconds | Time until the whole model was built in the simulation. **Lower is better.** The main result. |
| Planned completion time | simulated seconds | The plan's own estimate right after assignment, ignoring traffic and avoidance manoeuvres (`OptimisticMakespan`). This is what the MILP and the metaheuristics try to minimize. The gap to the real completion time shows what movement and congestion cost. |
| Smallest gap between robots | meters | **Safety.** The closest two free robots came, surface to surface. **Negative means they overlapped.** The completion time cannot see this, so read the two together. |
| Steps with overlapping robots | simulation steps | **Safety.** How long robots overlapped (one step is 1/40 s). The second number also counts transport teams, drawn as bounding circles, so it can overstate contact. |
| Position swaps | count | **Safety.** How often two robots instantly exchanged positions to unblock a team: a teleport a real robot could not do. |
| Top speed | m/s | The fastest agent, leaving out swaps. The limit is 4 m/s. |
| Assignment time | computer seconds | Real time spent deciding who does what (`AssignmentTime` in the simulator's file). |
| Preparation / Execution time | computer seconds | Real time until the assignment is done (`PreExecutionRuntime`) / spent simulating movement (`ExecutionRuntime`). |
| Total computer time | computer seconds | Everything, measured by this notebook. |
| Valid MILP solution | yes / no | Whether the MILP solver returned an assignment (MILP methods only). |
| Solver outcome | status, gap, bound | How far the solver got (MILP methods only). `TIME_LIMIT` with a gap means the schedule is not proven optimal. |
| Search | schedules | How many candidate schedules a metaheuristic evaluated, and whether it stopped at the time limit. |
| Code version | hash, commit | Which code produced this result, so it can be traced and reproduced. |

**Simulated time is not computer time.** Simulated seconds describe the robots' world; computer seconds depend on
your computer and on whether this was the first run after a restart (compilation). A dash (—) means *not available*,
for example because the run failed or the method does not use a solver.

In [29]:
OPEN_ANIMATION_IN_BROWSER = false     # true = also open the saved animation file in your web browser

show_visuals(LAST_RUN)                # result table, planned schedule, search progress, 3-D animation
if OPEN_ANIMATION_IN_BROWSER && LAST_RUN !== nothing && LAST_RUN[:animation_file] !== missing
    open_in_browser(LAST_RUN[:animation_file])
end;

Item,Value,Meaning / units
Status,SUCCESS,project completed
Run ID,20261103-233318-247_tractor_greedy_rovk,
Model,Tractor,10 robots (model default)
Assignment method,Greedy,Heuristic rule; solver: not used
Movement profile,Recommended,"RVO2 on, Tangent Bug on, Dispersion on"
Random seed,1,
Completion time (makespan),22.75 s,SIMULATED time until the model was fully built
Planned completion time,14.88 s,"SIMULATED; the schedule's estimate after assignment, ignoring traffic (""OptimisticMakespan"")"
Smallest gap between robots,0.017 m,"SAFETY; the closest two free robots came, surface to surface. Negative means they overlapped"
Steps with overlapping robots,0,"SAFETY; 2 when transport teams count too (as bounding circles, so conservative)"


HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"900\" height=\"258\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"900\" height=\"258\" fill=\"white\"/><text x=\"12\" y=\"20\" font-size=\"14\" font-weight=\"bold\" fill=\"#222\">Planned schedule: Tractor, 10 robots | Greedy | Recommended</text><text x=\"12\" y=\"37\" fill=\"#555\">27 transport tasks · 10 robots · planned completion 14.88 s</text><rect x=\"92\" y=\"50\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"61\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 1</text><text x=\"86\" y=\"76\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 2</text><rect x=\"92\" y=\"80\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"91\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 3</text><text x=\"86\" y=\"106\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 4</text><rect x=\"92\" y=\"110\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"121\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 5</text><text x=\"86\" y=\"136\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 6</text><rect x=\"92\" y=\"140\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"151\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 7</text><text x=\"86\" y=\"166\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 8</text><rect x=\"92\" y=\"170\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"181\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 9</text><text x=\"86\" y=\"196\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 10</text><rect x=\"101.64335664335664\" y=\"127\" width=\"125.36363636363637\" height=\"11\" fill=\"#4e79a7\"><title>robot 6 → part 17: sets off 0.18 s, team forms 0.18 s, delivered 2.45 s</title></rect><rect x=\"103.02097902097903\" y=\"67\" width=\"90.92307692307692\" height=\"11\" fill=\"#4e79a7\"><title>robot 2 → part 18: sets off 0.20 s, team forms 0.20 s, delivered 1.85 s</title></rect><rect x=\"111.28671328671328\" y=\"112\" width=\"176.33566433566432\" height=\"11\" fill=\"#4e79a7\"><title>robot 5 → part 15: sets off 0.35 s, team forms 0.35 s, delivered 3.55 s</title></rect><rect x=\"115.41958041958043\" y=\"52\" width=\"96.43356643356645\" height=\"11\" fill=\"#4e79a7\"><title>robot 1 → part 13: sets off 0.43 s, team forms 0.43 s, delivered 2.18 s</title></rect><rect x=\"118.17482517482517\" y=\"172\" width=\"104.6993006993007\" height=\"11\" fill=\"#4e79a7\"><title>robot 9 → part 11: sets off 0.48 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"116.7972027972028\" y=\"97\" width=\"1.377622377622373\" height=\"11\" fill=\"#c9d1d4\"><title>robot 4 → part 11: sets off 0.45 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"118.17482517482517\" y=\"97\" width=\"104.6993006993007\" height=\"11\" fill=\"#4e79a7\"><title>robot 4 → part 11: sets off 0.45 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"129.19580419580421\" y=\"157\" width=\"78.52447552447552\" height=\"11\" fill=\"#4e79a7\"><title>robot 8 → part 10: sets off 0.68 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"115.41958041958043\" y=\"82\" width=\"13.776223776223787\" height=\"11\" fill=\"#c9d1d4\"><title>robot 3 → part 10: sets off 0.43 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"129.19580419580421\" y=\"82\" width=\"78.52447552447552\" height=\"11\" fill=\"#4e79a7\"><title>robot 3 → part 10: sets off 0.43 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"133.32867132867133\" y=\"187\" width=\"84.03496503496504\" height=\"11\" fill=\"#4e79a7\"><title>robot 10 → part 12: sets off 0.75 s, team forms 0.75 s, delivered 2.28 s</title></rect><rect x=\"105.77622377622377\" y=\"142\" width=\"27.55244755244756\" height=\"11\" fill=\"#c9d1d4\"><title>robot 7 → part 12: sets off 0.25 s, team forms 0.75 s, delivered 2.28 s</title></rect><rect x=\"

Item,Value
Run,"20261103-233318-247_tractor_greedy_rovk | Tractor, 10 robots | Greedy | Recommended | seed 1"
File,results\notebook_lab\runs\20261103-233318-247_tractor_greedy_rovk\simulator_output\greedy_RVO_Dispersion_TangentBug\20261103_233319_visualization.html
File size,4.4 MB
"Moving objects (robots, parts, transport teams)",64
Animated objects / keyframes,263 / 24499
Animation length,36.0 s at 30 frames per second (1 frame = 1 simulation step)


HTML{String}("<iframe srcdoc=\"&lt;!doctype html&gt;\n&lt;!--\n  Animation player for notebooks/ConstructionBots_Experiment_Lab.ipynb.\n  Filled in by notebook_helpers.jl (_player_page), which replaces the double-underscore\n  placeholders below: the title, the scene frame&#39;s attribute (data-b64 = the whole saved\n  file in base64, or src = its URL on the kernel&#39;s viewer server) and the raw-file link.\n  (Do not write the placeholder names in this comment: every occurrence is replaced.)\n  It shows the run&#39;s saved MeshCat file, unchanged, in a frame and\n  drives MeshCat&#39;s own animation player (viewer.animator) from the controls\n  below it. Adapted from dashboard/simulation/player.html.\n--&gt;\n&lt;html lang=&quot;en&quot;&gt;\n&lt;head&gt;\n&lt;meta charset=&quot;utf-8&quot;&gt;\n&lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1&quot;&gt;\n&lt;title&gt;Experiment Lab animation&lt;/title&gt;\n&lt;style&gt;\n  * { box-sizing: border-box; }\n  html, body { margin: 0; height: 100%; background: #F5F5F2; color: #202A32;\n    font: 13px/1.4 &quot;Segoe UI&quot;, system-ui, -apple-system, &quot;Helvetica Neue&quot;, Arial, sans-serif; }\n  .shell { display: flex; flex-direction: column; height: 100%; }\n  .stage { position: relative; flex: 1 1 auto; min-height: 200px; overflow: hidden;\n    background: linear-gradient(#F5F5F2, #D9DFE1); }\n  iframe { position: absolute; inset: 0; width: 100%; height: 100%; border: 0; display: block; }\n  .overlay { position: absolute; inset: 0; display: flex; align-items: center; justify-content: center;\n    padding: 24px; text-align: center; color: #687581; background: linear-gradient(#F5F5F2, #E3E8E9); }\n  .hidden { display: none !important; }\n  .bar { display: flex; flex-wrap: wrap; align-items: center; gap: 6px; padding: 6px 8px; background: #fff;\n    border-top: 1px solid #DDE2E4; }\n  button, select { font: inherit; color: #202A32; background: #fff; border: 1px solid #DDE2E4; border-radius: 5px;\n    padding: 3px 9px; cursor: pointer; }\n  button:hover, select:hover { border-color: #456A8A; }\n  button.primary { background: #147D83; border-color: #147D83; color: #fff; min-width: 60px; }\n  button:disabled, select:disabled { opacity: .45; cursor: default; }\n  input[type=range] { flex: 1 1 140px; min-width: 100px; accent-color: #147D83; }\n  .time { font-variant-numeric: tabular-nums; color: #687581; min-width: 110px; }\n  .title { padding: 3px 8px; color: #687581; font-size: 12px; background: #fff; white-space: nowrap;\n    overflow: hidden; text-overflow: ellipsis; }\n&lt;/style&gt;\n&lt;/head&gt;\n&lt;body&gt;\n&lt;div class=&quot;shell&quot;&gt;\n  &lt;div class=&quot;title&quot; id=&quot;title&quot;&gt;20261103-233318-247_tractor_greedy_rovk | Tractor, 10 robots | Greedy | Recommended | seed 1&lt;/div&gt;\n  &lt;div class=&quot;stage&quot;&gt;\n    &lt;iframe id=&quot;scene&quot; title=&quot;3D animation of the simulation&quot; data-b64=&quot;ICAgIDwhRE9DVFlQRSBodG1sPgogICAgPGh0bWw+CiAgICAgICAgPGhlYWQ+IDxtZXRhIGNoYXJzZXQ9dXRmLTg+IDx0aXRsZT5NZXNoQ2F0PC90aXRsZT4gPC9oZWFkPgogICAgICAgIDxib2R5PgogICAgICAgICAgICA8ZGl2IGlkPSJtZXNoY2F0LXBhbmUiPgogICAgICAgICAgICA8L2Rpdj4KICAgICAgICAgICAgPHNjcmlwdD4KICAgICAgICAgICAgICAgIC8qISBGb3IgbGljZW5zZSBpbmZvcm1hdGlvbiBwbGVhc2Ugc2VlIG1haW4ubWluLmpzLkxJQ0VOU0UudHh0ICovCiFmdW5jdGlvbihBLHQpeyJvYmplY3QiPT10eXBlb2YgZXhwb3J0cyYmIm9iamVjdCI9PXR5cGVvZiBtb2R1bGU/bW9kdWxlLmV4cG9ydHM9dCgpOiJmdW5jdGlvbiI9PXR5cGVvZiBkZWZpbmUmJmRlZmluZS5hbWQ/ZGVmaW5lKFtdLHQpOiJvYmplY3QiPT10eXBlb2YgZXhwb3J0cz9leHBvcnRzLk1lc2hDYXQ9dCgpOkEuTWVzaENhdD10KCl9KHNlbGYsKCgpPT4oKCk9Pnt2YXIgX193ZWJwYWNrX21vZHVsZXNfXz17Nzc1OihBLHQsZSk9Pnt2YXIgZztBPWUubm1kKEEpLGZ1bmN0aW9uKCl7aWYodm9pZCAwIT09QS5leHBvcnRzKXZhciBCPWUoNDkxKSxpPWUoNjI2KSxJPWUoMjU4KTt2YXIgUT17ZnVuY3Rpb246ITAsb2JqZWN0OiEwfTtmdW5jdGlvbiBvKEEpe3JldHVybiBBJiZBLk9iamVjdD09PU9iamVjdD9BOm51bGx9cGFyc2VGbG9hdCxwYXJzZUludDt2YXIgcz1RW3R5cGVvZiB0XSYmdCYmIXQubm9kZVR5cGU/dDp2b2lkIDAsRT1RLm9iamVjdCYmQSYm

## 5. Compare algorithms (benchmark)

A **comparison** runs several experiments one after another, changing **only the assignment algorithm** (or, in
presets C–E, one other kind of setting) and holding everything else fixed: same model, robots, seed, movement and
limits. The default is the **Custom** list in the next cell: nine algorithms covering all three families. Or
choose a preset:

| `COMPARISON_PRESET` | What is compared | Runs | Typical time (Tractor) |
|---|---|---|---|
| `"Custom"` | The algorithms in `ALGORITHMS_TO_COMPARE` | as listed | 2–4 min for the default list |
| `"A"` Baseline | Greedy + Recommended movement | 1 | under 1 min |
| `"B"` Greedy vs MILP | Greedy vs MILP vs Greedy + MILP warm start | 3 | about 2× the time limit |
| `"C"` Solver comparison | MILP with HiGHS vs GLPK vs Gurobi (skipped without a licence) | 3 | about 2× the time limit |
| `"D"` Movement ablation | Greedy with each of the 5 movement profiles | 5 | 1–2 min |
| `"E"` Scaling | Tractor with 6 / 10 / 14 robots, then X-Wing (mini) with its default 20 robots | 4 | 1–3 min |
| `"F"` Heuristic rules | Greedy vs Greedy with travel vs Most complex first vs Nearest station first | 4 | under 1 min |
| `"G"` Metaheuristics benchmark | Genetic algorithm vs Particle swarm vs Simulated annealing vs Random search (the control), with Greedy and Greedy with travel as references | 6 | 2–3 min |
| `"H"` All algorithms | all ten | 10 | about 3× the time limit |

`COMPARISON_MODEL` and `COMPARISON_ROBOTS` apply to every preset except E. Before the timed runs, `WARM_UP = true`
runs each *kind* of configuration once on the Tractor (short search, MILP limited to 30 s), untimed and unsaved,
so that one-time compilation does not distort the computer-time measurements.

**To stop a comparison early:** create an empty file named `STOP` in the batch folder printed at the start
(the current run finishes, the remaining ones are listed as *skipped*), or press the **Interrupt** button
(the current run is recorded as *interrupted*). Completed runs are always kept.

**One run per algorithm is a demonstration, not evidence.** Rankings on one seed are often reversed on the next. For
conclusions, use Section 6 (several matched seeds, with a statistical test).

> **Reading a movement ablation (preset D).** Switching avoidance methods **off** usually gives a *shorter*
> completion time (for example, Recommended 22.75 s vs No avoidance methods 14.93 s on the Tractor, seed 1).
> That is **not** an improvement: without avoidance, robots drive through each other and through building sites.
> The completion time does not penalize that, so compare movement profiles on completion time *and* on the
> safety columns (robot gap, overlap steps, swaps).

In [30]:
# =====================================================================
#  ✏️ EDIT ME: COMPARISON SETTINGS
# =====================================================================
RUN_COMPARISON                = true        # false = skip this section (useful with "Run All")
COMPARISON_PRESET             = "Custom"    # "Custom" = the list below, or a preset "A" to "H" (table above)
ALGORITHMS_TO_COMPARE         = ["Greedy", "Greedy with travel", "Most complex first", "Nearest station first",
                                 "Genetic algorithm", "Particle swarm", "Simulated annealing", "Random search",
                                 "Greedy + MILP warm start"]    # "MILP" from scratch may find no schedule in the time limit
COMPARISON_MODEL              = "Tractor"   # used by every preset except "E"
COMPARISON_ROBOTS             = nothing     # nothing = model default
COMPARISON_SEED               = 1
COMPARISON_TIME_LIMIT_SECONDS = 30          # per MILP / metaheuristic run
COMPARISON_SEARCH_ITERATIONS  = 25          # metaheuristics: the same budget for each of them
COMPARISON_SEARCH_POPULATION  = 16
COMPARISON_SAVE_ANIMATIONS    = false       # true = an animation for every run (slower; larger notebook)
WARM_UP                       = true

println("Comparison settings stored.")

Comparison settings stored.


In [31]:
COMPARISON_RESULTS = nothing     # forget any previous comparison first
if RUN_COMPARISON
    configs, description = preset_configs(COMPARISON_PRESET; model = COMPARISON_MODEL,
        num_robots = COMPARISON_ROBOTS, seed = COMPARISON_SEED, time_limit = COMPARISON_TIME_LIMIT_SECONDS,
        save_animation = COMPARISON_SAVE_ANIMATIONS, algorithms = ALGORITHMS_TO_COMPARE,
        search_iterations = COMPARISON_SEARCH_ITERATIONS, search_population = COMPARISON_SEARCH_POPULATION)
    println(description, "\n")
    COMPARISON_RESULTS = run_batch(configs; name = "preset-" * COMPARISON_PRESET, warm_up = WARM_UP,
                                   show_simulator_log = SHOW_SIMULATOR_LOG)
else
    println("Comparison skipped (RUN_COMPARISON = false).")
end;

Custom comparison: Greedy vs Greedy with travel vs Most complex first vs Nearest station first vs Genetic algorithm vs Particle swarm vs Simulated annealing vs Random search vs Greedy + MILP warm start. Model, robots, seed and movement are identical.

Batch 20261103-234041_presetcustom_dpoo: 9 run(s). Results folder:
  C:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl\results\notebook_lab\batches\20261103-234041_presetcustom_dpoo
To stop after the current run, create an empty file named STOP in that folder.

Warm-up: 9 untimed, unsaved Tractor run(s) to compile the code first. These are not part of the results.


Simulating...  46%|███████████████████████                           |  ETA: 0:00:01 ( 9.04 ms/it)
   step_num: 230
   n_closed: 157
4m    n_total: 289


Simulating...  73%|█████████████████████████████████████             |  ETA: 0:00:01 (11.33 ms/it)
   step_num: 571
   n_closed: 223
4m    n_total: 289


Simulating...  94%|███████████████████████████████████████████████   |  ETA: 0:00:00 (13.21 ms/it)
   step_num: 892
   n_closed: 274
4m    n_total: 289


Simulating... 100%|██████████████████████████████████████████████████| Time: 0:00:03 (12.53 ms/it)


  warm-up 1/9 (Greedy, no solver, Recommended): success, 6 s


Simulating...  37%|███████████████████                               |  ETA: 0:00:02 (11.04 ms/it)
   step_num: 188
   n_closed: 137
4m    n_total: 289


Simulating...  53%|███████████████████████████                       |  ETA: 0:00:02 (15.62 ms/it)
   step_num: 340
   n_closed: 176
4m    n_total: 289


Simulating...  84%|██████████████████████████████████████████        |  ETA: 0:00:01 (14.99 ms/it)
   step_num: 566
   n_closed: 249
4m    n_total: 289


Simulating... 100%|██████████████████████████████████████████████████| Time: 0:00:03 (14.23 ms/it)


  warm-up 2/9 (Greedy with travel, no solver, Recommended): success, 6 s


Simulating...  23%|████████████                                      |  ETA: 0:00:03 (17.95 ms/it)
   step_num: 223
   n_closed: 102
4m    n_total: 289


Simulating...  48%|█████████████████████████                         |  ETA: 0:00:02 (17.25 ms/it)
   step_num: 441
   n_closed: 163
4m    n_total: 289


Simulating...  76%|███████████████████████████████████████           |  ETA: 0:00:01 (16.46 ms/it)
   step_num: 692
   n_closed: 231
4m    n_total: 289


Simulating... 100%|██████████████████████████████████████████████████| Time: 0:00:03 (15.26 ms/it)


... (9 outputs left out)

Simulating...  45%|███████████████████████                           |  ETA: 0:00:01 ( 9.12 ms/it)
   step_num: 315
   n_closed: 156
4m    n_total: 289


Simulating...  70%|███████████████████████████████████               |  ETA: 0:00:01 (11.84 ms/it)
   step_num: 583
   n_closed: 216
4m    n_total: 289


Simulating... 100%|██████████████████████████████████████████████████| Time: 0:00:02 (11.83 ms/it)


  warm-up 8/9 (Random search, no solver, Recommended): success, 5 s


Simulating...  58%|██████████████████████████████                    |  ETA: 0:00:01 ( 7.11 ms/it)
   step_num: 332
   n_closed: 187
4m    n_total: 289


Simulating...  79%|████████████████████████████████████████          |  ETA: 0:00:01 (10.52 ms/it)
   step_num: 634
   n_closed: 237
4m    n_total: 289


Simulating... 100%|██████████████████████████████████████████████████| Time: 0:00:02 (11.15 ms/it)


  warm-up 9/9 (Greedy + MILP warm start, HiGHS, Recommended): success, 40 s

──────────────────────────────────────────────────────────────────────────────
Run 1 of 9: [Greedy] Tractor | Greedy | Recommended | seed 1
Run 20261103-234206-255_tractor_greedy_smrr
  Reused the saved result of run 20261103-233318-247_tractor_greedy_rovk (same settings and code). Nothing was simulated.
  Status: SUCCESS (reused the saved result of run 20261103-233318-247_tractor_greedy_rovk; nothing was simulated again)
  Elapsed batch time: 0.2 s
──────────────────────────────────────────────────────────────────────────────
Run 2 of 9: [Greedy with travel] Tractor | Greedy with travel | Recommended | seed 1
Run 20261103-234206-503_tractor_greedy_with_travel_esyb
  Reused the saved result of run 20261103-121309-212_tractor_greedy_with_travel_bpqc (same settings and code). Nothing was simulated.
  Status: SUCCESS (reused the saved result of run 20261103-121309-212_tractor_greedy_with_travel_bpqc; nothing was 

### Comparison charts

The next cell shows the results table and these charts (each is also saved as an `.svg` file in the batch folder):

* **Completion time in the simulation**: the main result. Each bar is one configuration, coloured by algorithm. Check
  the safety columns of the table too: a faster run with overlapping robots is not a better one.
* **Planned completion time**: what each algorithm *planned*, before traffic. Comparing the two charts shows
  whether a better plan also gave a faster build.
* **Assignment computer time**: what each algorithm cost in computer time.
* **Search progress**: the metaheuristics' improvement per iteration, with the other algorithms' planned
  completion times as dashed reference lines.
* **Planned schedules** of all runs on **one time axis**, so you can see directly which plan is tighter.

Runs that did not succeed get **no bar**; their status is written in red instead (a failed run is never drawn as zero).

In [32]:
show_comparison(COMPARISON_RESULTS)

Label,Status,Model,Robots,Assignment,Solver,Movement,Seed,Completion time [sim s],Planned [sim s],Robot gap [m],Overlap steps,Swaps,Assignment [cpu s],Execution [cpu s],Total [cpu s]
Greedy,SUCCESS,Tractor,10,Greedy,not used,Recommended,1,22.75,14.88,0.017,0,0,0.20,3.50,0.00
Greedy with travel,SUCCESS,Tractor,10,Greedy with travel,not used,Recommended,1,18.32,12.47,0.017,0,0,0.61,12.40,0.00
Most complex first,SUCCESS,Tractor,10,Most complex first,not used,Recommended,1,23.75,16.55,-0.030,2,0,0.65,24.31,0.00
Nearest station first,SUCCESS,Tractor,10,Nearest station first,not used,Recommended,1,24.53,19.43,-0.015,2,0,0.60,14.46,0.00
Genetic algorithm,SUCCESS,Tractor,10,Genetic algorithm,not used,Recommended,1,18.53,11.23,-0.004,1,0,6.80,4.88,0.00
Particle swarm,SUCCESS,Tractor,10,Particle swarm,not used,Recommended,1,19.50,11.88,0.020,0,0,7.89,5.27,0.00
Simulated annealing,SUCCESS,Tractor,10,Simulated annealing,not used,Recommended,1,19.15,11.45,0.020,0,0,6.83,5.82,0.00
Random search,SUCCESS,Tractor,10,Random search,not used,Recommended,1,17.70,12.30,0.016,0,0,6.39,4.03,0.00
Greedy + MILP warm start,SUCCESS,Tractor,10,Greedy + MILP warm start,HiGHS,Recommended,1,22.75,14.88,0.017,0,0,30.48,3.42,0.00


HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"1110\" height=\"360\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"1110\" height=\"360\" fill=\"white\"/><text x=\"555.0\" y=\"22\" text-anchor=\"middle\" font-size=\"15\" font-weight=\"bold\">Completion time in the simulation (lower is better)</text><line x1=\"80\" x2=\"1090\" y1=\"250.0\" y2=\"250.0\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"254.0\" text-anchor=\"end\" fill=\"#444\">0.00</text><line x1=\"80\" x2=\"1090\" y1=\"235.10836324956787\" y2=\"235.10836324956787\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"239.10836324956787\" text-anchor=\"end\" fill=\"#444\">2.00</text><line x1=\"80\" x2=\"1090\" y1=\"220.21672649913575\" y2=\"220.21672649913575\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"224.21672649913575\" text-anchor=\"end\" fill=\"#444\">4.00</text><line x1=\"80\" x2=\"1090\" y1=\"205.32508974870362\" y2=\"205.32508974870362\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"209.32508974870362\" text-anchor=\"end\" fill=\"#444\">6.00</text><line x1=\"80\" x2=\"1090\" y1=\"190.4334529982715\" y2=\"190.4334529982715\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"194.4334529982715\" text-anchor=\"end\" fill=\"#444\">8.00</text><line x1=\"80\" x2=\"1090\" y1=\"175.5418162478394\" y2=\"175.5418162478394\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"179.5418162478394\" text-anchor=\"end\" fill=\"#444\">10.00</text><line x1=\"80\" x2=\"1090\" y1=\"160.65017949740724\" y2=\"160.65017949740724\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"164.65017949740724\" text-anchor=\"end\" fill=\"#444\">12.00</text><line x1=\"80\" x2=\"1090\" y1=\"145.75854274697514\" y2=\"145.75854274697514\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"149.75854274697514\" text-anchor=\"end\" fill=\"#444\">14.00</text><line x1=\"80\" x2=\"1090\" y1=\"130.866905996543\" y2=\"130.866905996543\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"134.866905996543\" text-anchor=\"end\" fill=\"#444\">16.00</text><line x1=\"80\" x2=\"1090\" y1=\"115.97526924611088\" y2=\"115.97526924611088\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"119.97526924611088\" text-anchor=\"end\" fill=\"#444\">18.00</text><line x1=\"80\" x2=\"1090\" y1=\"101.08363249567876\" y2=\"101.08363249567876\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"105.08363249567876\" text-anchor=\"end\" fill=\"#444\">20.00</text><line x1=\"80\" x2=\"1090\" y1=\"86.19199574524663\" y2=\"86.19199574524663\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"90.19199574524663\" text-anchor=\"end\" fill=\"#444\">22.00</text><line x1=\"80\" x2=\"1090\" y1=\"71.3003589948145\" y2=\"71.3003589948145\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"75.3003589948145\" text-anchor=\"end\" fill=\"#444\">24.00</text><line x1=\"80\" x2=\"1090\" y1=\"56.4087222443824\" y2=\"56.4087222443824\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"60.4087222443824\" text-anchor=\"end\" fill=\"#444\">26.00</text><line x1=\"80\" x2=\"1090\" y1=\"41.517085493950276\" y2=\"41.517085493950276\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"45.517085493950276\" text-anchor=\"end\" fill=\"#444\">28.00</text><text transform=\"translate(18,145.0) rotate(-90)\" text-anchor=\"middle\" fill=\"#222\">Completion time [simulated s]</text><rect x=\"102.44444444444446\" y=\"80.60763196383459\" width=\"67.33333333333333\" height=\"169.3923680361654\" fill=\"#4e79a7\"/><text x=\"136.11111111111111\" y=\"74.60763196383459\" text-anchor=\"middle\" fill=\"#222\">22.75 </text><text x=\"136.11111111111111\" y=\"266\" text-anchor=\"middle\" fill=\"#222\">Greedy</text><rect x=\"214.66666666666669\" y=\"113.55537827416566\" width=\"67.33333333333333\" height=\"136.44462172583434\" fill=\"#86bcb6\"/><text x=\"248.33333333333334\" y=\"107.55537827416566\" text-anchor=\"middle\" fill=\"#222\">18.32 </text><text x=\"248.33333333333334\" y=\"266\" text-anchor=\"middle\" fill=\"#222\">Greedy with</text><text x=\"248.33333333333334\" y=\"280\" text-anchor=\"middle\" fill=\"#222\">travel</text><rect x=\"326.88888888888886\" y=\"73.16181358861854\" width=\"67.33333333333

HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"1110\" height=\"360\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"1110\" height=\"360\" fill=\"white\"/><text x=\"555.0\" y=\"22\" text-anchor=\"middle\" font-size=\"15\" font-weight=\"bold\">Planned completion time right after assignment (lower is better)</text><line x1=\"80\" x2=\"1090\" y1=\"250.0\" y2=\"250.0\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"254.0\" text-anchor=\"end\" fill=\"#444\">0.00</text><line x1=\"80\" x2=\"1090\" y1=\"231.19858989424208\" y2=\"231.19858989424208\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"235.19858989424208\" text-anchor=\"end\" fill=\"#444\">2.00</text><line x1=\"80\" x2=\"1090\" y1=\"212.39717978848415\" y2=\"212.39717978848415\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"216.39717978848415\" text-anchor=\"end\" fill=\"#444\">4.00</text><line x1=\"80\" x2=\"1090\" y1=\"193.5957696827262\" y2=\"193.5957696827262\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"197.5957696827262\" text-anchor=\"end\" fill=\"#444\">6.00</text><line x1=\"80\" x2=\"1090\" y1=\"174.7943595769683\" y2=\"174.7943595769683\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"178.7943595769683\" text-anchor=\"end\" fill=\"#444\">8.00</text><line x1=\"80\" x2=\"1090\" y1=\"155.99294947121035\" y2=\"155.99294947121035\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"159.99294947121035\" text-anchor=\"end\" fill=\"#444\">10.00</text><line x1=\"80\" x2=\"1090\" y1=\"137.19153936545243\" y2=\"137.19153936545243\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"141.19153936545243\" text-anchor=\"end\" fill=\"#444\">12.00</text><line x1=\"80\" x2=\"1090\" y1=\"118.3901292596945\" y2=\"118.3901292596945\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"122.3901292596945\" text-anchor=\"end\" fill=\"#444\">14.00</text><line x1=\"80\" x2=\"1090\" y1=\"99.58871915393658\" y2=\"99.58871915393658\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"103.58871915393658\" text-anchor=\"end\" fill=\"#444\">16.00</text><line x1=\"80\" x2=\"1090\" y1=\"80.78730904817866\" y2=\"80.78730904817866\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"84.78730904817866\" text-anchor=\"end\" fill=\"#444\">18.00</text><line x1=\"80\" x2=\"1090\" y1=\"61.98589894242073\" y2=\"61.98589894242073\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"65.98589894242073\" text-anchor=\"end\" fill=\"#444\">20.00</text><line x1=\"80\" x2=\"1090\" y1=\"43.18448883666278\" y2=\"43.18448883666278\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"47.18448883666278\" text-anchor=\"end\" fill=\"#444\">22.00</text><text transform=\"translate(18,145.0) rotate(-90)\" text-anchor=\"middle\" fill=\"#222\">Planned completion [simulated s]</text><rect x=\"102.44444444444446\" y=\"110.1645123384254\" width=\"67.33333333333333\" height=\"139.8354876615746\" fill=\"#4e79a7\"/><text x=\"136.11111111111111\" y=\"104.1645123384254\" text-anchor=\"middle\" fill=\"#222\">14.88 </text><text x=\"136.11111111111111\" y=\"266\" text-anchor=\"middle\" fill=\"#222\">Greedy</text><rect x=\"214.66666666666669\" y=\"132.72620446533494\" width=\"67.33333333333333\" height=\"117.27379553466506\" fill=\"#86bcb6\"/><text x=\"248.33333333333334\" y=\"126.72620446533494\" text-anchor=\"middle\" fill=\"#222\">12.47 </text><text x=\"248.33333333333334\" y=\"266\" text-anchor=\"middle\" fill=\"#222\">Greedy with</text><text x=\"248.33333333333334\" y=\"280\" text-anchor=\"middle\" fill=\"#222\">travel</text><rect x=\"326.88888888888886\" y=\"94.41833137485315\" width=\"67.33333333333333\" height=\"155.58166862514685\" fill=\"#76b7b2\"/><text x=\"360.55555555555554\" y=\"88.41833137485315\" text-anchor=\"middle\" fill=\"#222\">16.55 </text><text x=\"360.55555555555554\" y=\"266\" text-anchor=\"middle\" fill=\"#222\">Most complex</text><text x=\"360.55555555555554\" y=\"280\" text-anchor=\"middle\" fill=\"#222\">first</text><rect x=\"439.11111111111114\" y=\"67.3913043478261\" width=\"67.33333333333333\" height=\"182.6086956521739\" fill=\"#59a14f\"/><text x=\"472.7777777777778\" y=\"61.39130434782609\" text-a

HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"1110\" height=\"360\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"1110\" height=\"360\" fill=\"white\"/><text x=\"555.0\" y=\"22\" text-anchor=\"middle\" font-size=\"15\" font-weight=\"bold\">Computer time spent on task assignment</text><line x1=\"80\" x2=\"1090\" y1=\"250.0\" y2=\"250.0\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"254.0\" text-anchor=\"end\" fill=\"#444\">0.00</text><line x1=\"80\" x2=\"1090\" y1=\"238.01937446409872\" y2=\"238.01937446409872\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"242.01937446409872\" text-anchor=\"end\" fill=\"#444\">2.00</text><line x1=\"80\" x2=\"1090\" y1=\"226.0387489281974\" y2=\"226.0387489281974\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"230.0387489281974\" text-anchor=\"end\" fill=\"#444\">4.00</text><line x1=\"80\" x2=\"1090\" y1=\"214.05812339229612\" y2=\"214.05812339229612\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"218.05812339229612\" text-anchor=\"end\" fill=\"#444\">6.00</text><line x1=\"80\" x2=\"1090\" y1=\"202.0774978563948\" y2=\"202.0774978563948\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"206.0774978563948\" text-anchor=\"end\" fill=\"#444\">8.00</text><line x1=\"80\" x2=\"1090\" y1=\"190.09687232049353\" y2=\"190.09687232049353\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"194.09687232049353\" text-anchor=\"end\" fill=\"#444\">10.00</text><line x1=\"80\" x2=\"1090\" y1=\"178.11624678459225\" y2=\"178.11624678459225\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"182.11624678459225\" text-anchor=\"end\" fill=\"#444\">12.00</text><line x1=\"80\" x2=\"1090\" y1=\"166.13562124869094\" y2=\"166.13562124869094\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"170.13562124869094\" text-anchor=\"end\" fill=\"#444\">14.00</text><line x1=\"80\" x2=\"1090\" y1=\"154.15499571278966\" y2=\"154.15499571278966\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"158.15499571278966\" text-anchor=\"end\" fill=\"#444\">16.00</text><line x1=\"80\" x2=\"1090\" y1=\"142.17437017688837\" y2=\"142.17437017688837\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"146.17437017688837\" text-anchor=\"end\" fill=\"#444\">18.00</text><line x1=\"80\" x2=\"1090\" y1=\"130.19374464098706\" y2=\"130.19374464098706\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"134.19374464098706\" text-anchor=\"end\" fill=\"#444\">20.00</text><line x1=\"80\" x2=\"1090\" y1=\"118.21311910508578\" y2=\"118.21311910508578\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"122.21311910508578\" text-anchor=\"end\" fill=\"#444\">22.00</text><line x1=\"80\" x2=\"1090\" y1=\"106.23249356918447\" y2=\"106.23249356918447\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"110.23249356918447\" text-anchor=\"end\" fill=\"#444\">24.00</text><line x1=\"80\" x2=\"1090\" y1=\"94.25186803328319\" y2=\"94.25186803328319\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"98.25186803328319\" text-anchor=\"end\" fill=\"#444\">26.00</text><line x1=\"80\" x2=\"1090\" y1=\"82.2712424973819\" y2=\"82.2712424973819\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"86.2712424973819\" text-anchor=\"end\" fill=\"#444\">28.00</text><line x1=\"80\" x2=\"1090\" y1=\"70.2906169614806\" y2=\"70.2906169614806\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"74.2906169614806\" text-anchor=\"end\" fill=\"#444\">30.00</text><line x1=\"80\" x2=\"1090\" y1=\"58.30999142557931\" y2=\"58.30999142557931\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"62.30999142557931\" text-anchor=\"end\" fill=\"#444\">32.00</text><line x1=\"80\" x2=\"1090\" y1=\"46.329365889678\" y2=\"46.329365889678\" stroke=\"#e5e5e5\"/><text x=\"74\" y=\"50.329365889678\" text-anchor=\"end\" fill=\"#444\">34.00</text><text transform=\"translate(18,145.0) rotate(-90)\" text-anchor=\"middle\" fill=\"#222\">Assignment time [computer s]</text><rect x=\"102.44444444444446\" y=\"248.7779762181893\" width=\"67.33333333333333\" height=\"1.222023781810691\" fill=\"#4e79a7\"/><text x=\"136.11111111111111\" y=\"242.7779762181893\" text-anchor=\"middle\" fill=\"#222\">0.20 </text><text x=\"136.11111111111111\" y=\"266\" text-anchor=\"m

HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"780\" height=\"380\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"780\" height=\"380\" fill=\"white\"/><text x=\"70\" y=\"22\" font-size=\"15\" font-weight=\"bold\" fill=\"#222\">Search progress: best planned completion time per iteration</text><line x1=\"70\" x2=\"550\" y1=\"306.6512588512982\" y2=\"306.6512588512982\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"310.6512588512982\" text-anchor=\"end\" fill=\"#444\">11.00</text><line x1=\"70\" x2=\"550\" y1=\"278.5238001573565\" y2=\"278.5238001573565\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"282.5238001573565\" text-anchor=\"end\" fill=\"#444\">12.00</text><line x1=\"70\" x2=\"550\" y1=\"250.3963414634147\" y2=\"250.3963414634147\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"254.3963414634147\" text-anchor=\"end\" fill=\"#444\">13.00</text><line x1=\"70\" x2=\"550\" y1=\"222.26888276947295\" y2=\"222.26888276947295\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"226.26888276947295\" text-anchor=\"end\" fill=\"#444\">14.00</text><line x1=\"70\" x2=\"550\" y1=\"194.14142407553118\" y2=\"194.14142407553118\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"198.14142407553118\" text-anchor=\"end\" fill=\"#444\">15.00</text><line x1=\"70\" x2=\"550\" y1=\"166.0139653815894\" y2=\"166.0139653815894\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"170.0139653815894\" text-anchor=\"end\" fill=\"#444\">16.00</text><line x1=\"70\" x2=\"550\" y1=\"137.88650668764765\" y2=\"137.88650668764765\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"141.88650668764765\" text-anchor=\"end\" fill=\"#444\">17.00</text><line x1=\"70\" x2=\"550\" y1=\"109.75904799370588\" y2=\"109.75904799370588\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"113.75904799370588\" text-anchor=\"end\" fill=\"#444\">18.00</text><line x1=\"70\" x2=\"550\" y1=\"81.6315892997641\" y2=\"81.6315892997641\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"85.6315892997641\" text-anchor=\"end\" fill=\"#444\">19.00</text><line x1=\"70\" x2=\"550\" y1=\"53.504130605822354\" y2=\"53.504130605822354\" stroke=\"#e5e5e5\"/><text x=\"64\" y=\"57.504130605822354\" text-anchor=\"end\" fill=\"#444\">20.00</text><text x=\"70.0\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">0</text><text x=\"108.4\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">2</text><text x=\"146.8\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">4</text><text x=\"185.2\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">6</text><text x=\"223.6\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">8</text><text x=\"262.0\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">10</text><text x=\"300.4\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">12</text><text x=\"338.8\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">14</text><text x=\"377.2\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">16</text><text x=\"415.59999999999997\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">18</text><text x=\"454.0\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">20</text><text x=\"492.4\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">22</text><text x=\"530.8\" y=\"344\" text-anchor=\"middle\" fill=\"#444\">24</text><line x1=\"70\" x2=\"550\" y1=\"328\" y2=\"328\" stroke=\"#222\"/><text x=\"310.0\" y=\"368\" text-anchor=\"middle\" fill=\"#222\">iteration (0 = start: Greedy and its task order)</text><text transform=\"translate(18,185.0) rotate(-90)\" text-anchor=\"middle\" fill=\"#222\">planned completion [simulated s]</text><line x1=\"70\" x2=\"550\" y1=\"197.65735641227388\" y2=\"197.65735641227388\" stroke=\"#4e79a7\" stroke-width=\"1.5\" stroke-dasharray=\"6 4\"><title>Greedy: 14.88 s</title></line><line x1=\"70\" x2=\"550\" y1=\"265.16325727773415\" y2=\"265.16325727773415\" stroke=\"#86bcb6\" stroke-width=\"1.5\" stroke-dasharray=\"6 4\"><title>Greedy with travel: 12.47 s</title></line><line x1=\"70\" x2=\"550\" y1=\"150.54386309992142\" y2=\"150.54386309992142\" stroke=\"#76b7b2\" stroke-width=\"1.5\" stroke-dasharray=\"6 4\"><title>Most complex first: 16.55 s</title></l

HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"900\" height=\"2130\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"900\" height=\"2130\" fill=\"white\"/><text x=\"12\" y=\"20\" font-size=\"14\" font-weight=\"bold\" fill=\"#222\">Planned schedule: Greedy</text><text x=\"12\" y=\"37\" fill=\"#555\">27 transport tasks · 10 robots · planned completion 14.88 s</text><rect x=\"92\" y=\"50\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"61\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 1</text><text x=\"86\" y=\"76\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 2</text><rect x=\"92\" y=\"80\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"91\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 3</text><text x=\"86\" y=\"106\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 4</text><rect x=\"92\" y=\"110\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"121\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 5</text><text x=\"86\" y=\"136\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 6</text><rect x=\"92\" y=\"140\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"151\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 7</text><text x=\"86\" y=\"166\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 8</text><rect x=\"92\" y=\"170\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"181\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 9</text><text x=\"86\" y=\"196\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 10</text><rect x=\"99.14507772020725\" y=\"127\" width=\"92.88601036269428\" height=\"11\" fill=\"#4e79a7\"><title>robot 6 → part 17: sets off 0.18 s, team forms 0.18 s, delivered 2.45 s</title></rect><rect x=\"100.16580310880829\" y=\"67\" width=\"67.36787564766837\" height=\"11\" fill=\"#4e79a7\"><title>robot 2 → part 18: sets off 0.20 s, team forms 0.20 s, delivered 1.85 s</title></rect><rect x=\"106.29015544041451\" y=\"112\" width=\"130.65284974093262\" height=\"11\" fill=\"#4e79a7\"><title>robot 5 → part 15: sets off 0.35 s, team forms 0.35 s, delivered 3.55 s</title></rect><rect x=\"109.35233160621762\" y=\"52\" width=\"71.45077720207253\" height=\"11\" fill=\"#4e79a7\"><title>robot 1 → part 13: sets off 0.43 s, team forms 0.43 s, delivered 2.18 s</title></rect><rect x=\"111.39378238341969\" y=\"172\" width=\"77.57512953367872\" height=\"11\" fill=\"#4e79a7\"><title>robot 9 → part 11: sets off 0.48 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"110.37305699481865\" y=\"97\" width=\"1.0207253886010363\" height=\"11\" fill=\"#c9d1d4\"><title>robot 4 → part 11: sets off 0.45 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"111.39378238341969\" y=\"97\" width=\"77.57512953367872\" height=\"11\" fill=\"#4e79a7\"><title>robot 4 → part 11: sets off 0.45 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"119.55958549222798\" y=\"157\" width=\"58.18134715025907\" height=\"11\" fill=\"#4e79a7\"><title>robot 8 → part 10: sets off 0.68 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"109.35233160621762\" y=\"82\" width=\"10.207253886010363\" height=\"11\" fill=\"#c9d1d4\"><title>robot 3 → part 10: sets off 0.43 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"119.55958549222798\" y=\"82\" width=\"58.18134715025907\" height=\"11\" fill=\"#4e79a7\"><title>robot 3 → part 10: sets off 0.43 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"122.62176165803108\" y=\"187\" width=\"62.26424870466322\" height=\"11\" fill=\"#4e79a7\"><title>robot 10 → part 12: sets off 0.75 s, team forms 0.75 s, delivered 2.28 s</title></rect><rect x=\"102.20725388601036\" y=\"142\" width=\"20.414507772020713\" height=\"11\" fill=\"#c9d1d4\"><title>robot 7 → part 12: sets off 0.25 s, team forms 0.75 s, delivered 2.28 s</title></rect><rect x=\"122.62176165803108\" y=\"142\" w

Charts saved in C:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl\results\notebook_lab\batches\20261103-234041_presetcustom_dpoo


### Look at one compared run in detail

Shows the result table, schedule, search progress and (if `COMPARISON_SAVE_ANIMATIONS = true`) the animation of
one run of the comparison. `RUN_TO_SHOW = 1` is the first run in the table above.

In [33]:
RUN_TO_SHOW = 4      # ✏️ position in the comparison table (1 = first row)

if COMPARISON_RESULTS === nothing
    println("Run the comparison first.")
elseif !(1 <= RUN_TO_SHOW <= length(COMPARISON_RESULTS))
    println("RUN_TO_SHOW must be a number from 1 to ", length(COMPARISON_RESULTS), ".")
else
    show_visuals(COMPARISON_RESULTS[RUN_TO_SHOW])
end

Item,Value,Meaning / units
Status,SUCCESS,reused the saved result of run 20261103-121409-702_tractor_nearest_station_first_kvoe; nothing was simulated again
Run ID,20261103-234206-684_tractor_nearest_station_first_nceb,
Model,Tractor,10 robots (model default)
Assignment method,Nearest station first,Heuristic rule; solver: not used
Movement profile,Recommended,"RVO2 on, Tangent Bug on, Dispersion on"
Random seed,1,
Completion time (makespan),24.53 s,SIMULATED time until the model was fully built
Planned completion time,19.43 s,"SIMULATED; the schedule's estimate after assignment, ignoring traffic (""OptimisticMakespan"")"
Smallest gap between robots,-0.015 m,"SAFETY; the closest two free robots came, surface to surface. Negative means they overlapped"
Steps with overlapping robots,2,"SAFETY; 4 when transport teams count too (as bounding circles, so conservative)"


HTML{String}("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"900\" height=\"258\" font-family=\"sans-serif\" font-size=\"12\"><rect width=\"900\" height=\"258\" fill=\"white\"/><text x=\"12\" y=\"20\" font-size=\"14\" font-weight=\"bold\" fill=\"#222\">Planned schedule: Nearest station first</text><text x=\"12\" y=\"37\" fill=\"#555\">27 transport tasks · 10 robots · planned completion 19.43 s</text><rect x=\"92\" y=\"50\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"61\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 1</text><text x=\"86\" y=\"76\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 2</text><rect x=\"92\" y=\"80\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"91\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 3</text><text x=\"86\" y=\"106\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 4</text><rect x=\"92\" y=\"110\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"121\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 5</text><text x=\"86\" y=\"136\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 6</text><rect x=\"92\" y=\"140\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"151\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 7</text><text x=\"86\" y=\"166\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 8</text><rect x=\"92\" y=\"170\" width=\"788\" height=\"15\" fill=\"#f6f7f8\"/><text x=\"86\" y=\"181\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 9</text><text x=\"86\" y=\"196\" text-anchor=\"end\" fill=\"#444\" font-size=\"11\">robot 10</text><rect x=\"99.14507772020725\" y=\"127\" width=\"92.88601036269428\" height=\"11\" fill=\"#4e79a7\"><title>robot 6 → part 17: sets off 0.18 s, team forms 0.18 s, delivered 2.45 s</title></rect><rect x=\"100.16580310880829\" y=\"67\" width=\"67.36787564766837\" height=\"11\" fill=\"#4e79a7\"><title>robot 2 → part 18: sets off 0.20 s, team forms 0.20 s, delivered 1.85 s</title></rect><rect x=\"106.29015544041451\" y=\"112\" width=\"130.65284974093262\" height=\"11\" fill=\"#4e79a7\"><title>robot 5 → part 15: sets off 0.35 s, team forms 0.35 s, delivered 3.55 s</title></rect><rect x=\"109.35233160621762\" y=\"52\" width=\"71.45077720207253\" height=\"11\" fill=\"#4e79a7\"><title>robot 1 → part 13: sets off 0.43 s, team forms 0.43 s, delivered 2.18 s</title></rect><rect x=\"111.39378238341969\" y=\"172\" width=\"77.57512953367872\" height=\"11\" fill=\"#4e79a7\"><title>robot 9 → part 11: sets off 0.48 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"110.37305699481865\" y=\"97\" width=\"1.0207253886010363\" height=\"11\" fill=\"#c9d1d4\"><title>robot 4 → part 11: sets off 0.45 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"111.39378238341969\" y=\"97\" width=\"77.57512953367872\" height=\"11\" fill=\"#4e79a7\"><title>robot 4 → part 11: sets off 0.45 s, team forms 0.48 s, delivered 2.38 s</title></rect><rect x=\"119.55958549222798\" y=\"157\" width=\"58.18134715025907\" height=\"11\" fill=\"#4e79a7\"><title>robot 8 → part 10: sets off 0.68 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"109.35233160621762\" y=\"82\" width=\"10.207253886010363\" height=\"11\" fill=\"#c9d1d4\"><title>robot 3 → part 10: sets off 0.43 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"119.55958549222798\" y=\"82\" width=\"58.18134715025907\" height=\"11\" fill=\"#4e79a7\"><title>robot 3 → part 10: sets off 0.43 s, team forms 0.68 s, delivered 2.10 s</title></rect><rect x=\"167.53367875647666\" y=\"67\" width=\"27.559585492227995\" height=\"11\" fill=\"#c9d1d4\"><title>robot 2 → assembly 8: sets off 1.85 s, team forms 2.53 s, delivered 5.98 s</title></rect><rect x=\"195.09326424870466\" y=\"67\" width=\"140.86010362694304\" height=\"11\" fill=\"#f28e2b\"><title>robot 2 → assembly 8: sets off 1.85 s, team forms 2.53 s, delivered 5.98 s</title></rect><rect x=\"201.21761658031

No animation file for run 20261103-234206-684_tractor_nearest_station_first_nceb: SAVE_ANIMATION was false for it (or SAVE_RESULTS was false).


## 6. Repeated trials (this is where conclusions come from)

A single run can be lucky or unlucky. Repeating each configuration with several **random seeds** (different
robot starting positions) shows how stable a result is. Every configuration uses **the same seeds**, so the
comparison stays fair.

This section is **off by default** because it multiplies the running time (number of configurations × number
of seeds). Set `RUN_REPEATED_TRIALS = true` to use it. It reports:

* **Summary per configuration:** success rate; mean completion time of the successful runs with its 95 % confidence
  interval, standard deviation, minimum and maximum; and the safety record. Failed runs are counted, never averaged in.
* **Each configuration against Greedy, seed by seed:** the mean gain with its 95 % interval, the effect size, wins and
  losses, and a **paired sign-flip test**, corrected for several comparisons (Holm). A difference is only worth reporting
  if its interval excludes zero and the corrected p value is small.
* **Does the plan predict the simulation?** The rank correlation of planned and simulated completion time, per method.
  If it is near zero, a better plan does not mean a faster build.

**What to expect.** `docs/EXPERIMENTS.md` has this comparison over six models, up to 30 seeds each. Its main result is
that no method wins everywhere, and that the gain of the search depends on the model and the number of robots.

**How many seeds?** With fewer than 6 matched seeds the test cannot reach p < 0.05 at all, whatever the data. Use at
least 10, and 30 for anything you intend to publish. All of this is for **one** model: a result on the Tractor does
not carry over to other models by itself. For a batch over several models, use `scripts/method_comparison.jl`.

In [34]:
# ✏️ EDIT ME
RUN_REPEATED_TRIALS       = false
TRIALS_PRESET             = "Custom"      # "Custom" = TRIALS_ALGORITHMS, or any preset from Section 5
TRIALS_ALGORITHMS         = ["Greedy", "Greedy with travel", "Genetic algorithm", "Particle swarm",
                             "Simulated annealing", "Random search"]
TRIALS_MODEL              = "Tractor"
TRIALS_SEEDS              = 1:10          # matched seeds; 6 is the minimum for a test to reach p < 0.05
TRIALS_TIME_LIMIT_SECONDS = 30

TRIAL_RESULTS = nothing
if RUN_REPEATED_TRIALS
    base_configs, description = preset_configs(TRIALS_PRESET; model = TRIALS_MODEL,
                                               time_limit = TRIALS_TIME_LIMIT_SECONDS, algorithms = TRIALS_ALGORITHMS)
    configs = with_seeds(base_configs, TRIALS_SEEDS)
    println(description, "\nRepeated with seeds ", collect(TRIALS_SEEDS), ": ", length(configs), " runs.\n")
    TRIAL_RESULTS = run_batch(configs; name = "trials-" * TRIALS_PRESET, warm_up = true,
                              show_simulator_log = SHOW_SIMULATOR_LOG)
    show_summary(TRIAL_RESULTS; by = [:label])
    show_paired(TRIAL_RESULTS; baseline = "Greedy")
    show_plan_vs_simulation(TRIAL_RESULTS)
    folder = batch_folder(TRIAL_RESULTS)
    chart_metric(TRIAL_RESULTS, :makespan_s; title = "Completion time, mean and range over seeds",
                 ylabel = "Completion time [simulated s]", save_to = joinpath(folder, "chart_completion_time_trials.svg"))
    chart_metric(TRIAL_RESULTS, :optimistic_makespan_s; title = "Planned completion time, mean and range over seeds",
                 ylabel = "Planned completion [simulated s]", save_to = joinpath(folder, "chart_planned_trials.svg"))
else
    println("Repeated trials skipped (RUN_REPEATED_TRIALS = false).")
end

Repeated trials skipped (RUN_REPEATED_TRIALS = false).


## 7. Results files and CSV export

Everything is saved under `results/notebook_lab/` inside the repository (this folder is ignored by Git):

```
results/notebook_lab/
├── all_runs.csv                     one line per saved run, appended (never overwritten)
├── runs/<run ID>/                   one new folder per run
│   ├── effective_config.toml        your settings + the exact simulator arguments + software versions
│   ├── result.toml                  the result record shown in the tables, incl. the planned schedule
│   │                                and (metaheuristics) the search progress
│   ├── simulator_log.txt            the simulator's console output (solver log, progress messages)
│   ├── error.txt                    only if the run failed: the full technical error
│   └── simulator_output/
│       ├── variable_dump/           only if the run got stuck: the simulator's debugging dump (.jld2)
│       └── <mode>_<flags>/
│           ├── <date>_stats.toml    the simulator's own statistics file; Completed = false marks a run that crashed
│           └── <date>_visualization.html  the animation, if SAVE_ANIMATION = true
└── batches/<batch ID>/
    ├── summary.csv                  the comparison table
    ├── batch_info.toml              batch settings and software versions
    └── chart_*.svg                  the charts (completion, planned, assignment time, search, schedules)
```

The CSV columns are the same everywhere; empty cells mean *not available*. Times ending in `_s` are seconds:
`makespan_s` and `optimistic_makespan_s` are **simulated** seconds; `assignment_time_s`, `pre_execution_runtime_s`,
`execution_runtime_s` and `total_wall_time_s` are **computer** seconds. `assignment_family` is "Heuristic rule",
"Exact optimization" or "Metaheuristic"; the `search_*` columns describe a metaheuristic's budget and use.
`robot_min_gap_m`, `robot_overlap_steps`, `min_gap_m`, `overlap_steps`, `position_swaps` and `max_speed_mps` are the
safety record; `solver_status`, `solver_gap` and `solver_bound` say how far a MILP solver got; `source_hash` and
`git_commit` identify the code.
When a new notebook version adds columns, the old `all_runs.csv` is kept as `all_runs_before_<date>.csv`.

The next cell exports the latest comparison to a new, time-stamped CSV file (it never overwrites a file),
reads it back to confirm it is complete, and shows how many runs are in the overall log.

In [35]:
using Dates
records_to_export = something(TRIAL_RESULTS, COMPARISON_RESULTS,
                              LAST_RUN === nothing ? nothing : [LAST_RUN], Some(nothing))
if records_to_export === nothing
    println("Nothing to export yet.")
else
    export_path = joinpath(REPO_ROOT, "results", "notebook_lab", "exports",
                           "export_" * Dates.format(now(), "yyyymmdd-HHMMSS") * ".csv")
    export_csv(records_to_export, export_path)
    check = read_csv(export_path)
    println("Exported ", length(check), " run(s) with ", length(CSV_COLUMNS), " columns to:\n  ", export_path)
end

all_runs_file = joinpath(REPO_ROOT, "results", "notebook_lab", "all_runs.csv")
if isfile(all_runs_file)
    println("\nThe overall log ", all_runs_file, " contains ", length(read_csv(all_runs_file)), " saved run(s).")
end

Exported 9 run(s) with 52 columns to:
  c:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl\results\notebook_lab\exports\export_20261104-064845.csv

The overall log c:\Users\Ali Akarma\Documents\GitHub\ConstructionBots.jl\results\notebook_lab\all_runs.csv contains 43 saved run(s).


## 8. Troubleshooting

| What you see | What it means | What to do |
|---|---|---|
| `Settings problem: Unknown model "..."` (or method, solver, profile) | A name is misspelled. | Copy a name from the tables printed in Section 2. |
| Tables, charts or the animation appear only as plain text | The cell output cannot show HTML (for example a plain terminal). | Open the notebook in Jupyter, JupyterLab or VS Code, restart the kernel and run from the top. |
| `Settings problem: SEARCH_ITERATIONS ...` / `SEARCH_POPULATION ...` | The metaheuristic budget is out of range. | Iterations 1–1000, population 2–500, whole numbers. |
| The result says **STOPPED AT THE TIME LIMIT** (metaheuristics) | The search used all of `TIME_LIMIT_SECONDS` before finishing its budget. | Its best plan is still used. For a full search, raise the time limit or lower the budget. |
| The saved notebook file is very large | Animations are embedded in the outputs (a few MB each). | Use *Clear All Outputs* before saving or sharing the notebook. |
| `Settings problem: ... must be a whole number` / `true or false` | A value has the wrong form, e.g. `"10"` in quotes. | Numbers and `true`/`false` go **without** quotes; names go **in** quotes. |
| `GLPK cannot be combined with "Greedy + MILP warm start"` | The simulator cannot pass a starting solution to GLPK. | Use `SOLVER = "HiGHS"`. |
| `Solver Gurobi is not usable ... no valid Gurobi licence` | Gurobi is installed as a Julia package, but this computer has no licence. | Use HiGHS, or obtain a licence (Section 9 and `SETUP.md`). |
| `... is a large model ... set ALLOW_LARGE_MODELS = true` | Protection against accidentally starting a run of minutes to hours. | Set it in Advanced settings only if you mean it. |
| Status **FAILED**, "... robots are too few: the largest transport team ... needs N" | Some parts need a team of several robots. | Set `NUMBER_OF_ROBOTS` to at least N, or use `nothing`. |
| Status **FAILED**, "no feasible solution within the time limit" | The MILP solver found no assignment in time. | Increase `TIME_LIMIT_SECONDS`, or use Greedy / the warm-start method. |
| Status **INCOMPLETE** | Robots stopped making progress (or the step limit was reached) before the model was finished. Completion time is left empty on purpose. | Try another movement profile or seed; see the run's `simulator_log.txt`. |
| Environment check: `Python module rvo2: PROBLEM` | The simulator needs the Python RVO2 library for **every** run, even with RVO2 off. | Follow `SETUP.md`, step 3, then restart the kernel. |
| Environment check: `LDraw Parts Library: PROBLEM` | The part shapes are missing. | Install the library (`SETUP.md`, step 2) or set `LDRAW_LIBRARY_OVERRIDE`. |
| `UndefVarError: ... not defined` | A cell was run before the cells above it. | Restart the kernel and run from the top. |
| The first run takes minutes | One-time compilation after every restart. | Normal. Later runs are fast. |
| The animation shows the finished model but nothing moves; the check says **0 moving objects** | The file was saved by an older version of this code, which recorded no movement. | Run the experiment again. |
| The animation frame stays empty or says the page cannot be reached | Only for very large animations, which are served by the notebook's Julia session instead of being embedded; the session was restarted. | Run the cell again, or open the saved file directly (`OPEN_ANIMATION_IN_BROWSER = true`). |
| The animation shows robots passing through each other | With `SMOOTH_ANIMATION = false` the animation only updates when tasks finish, so robots are drawn on straight lines between updates. With "No avoidance methods" they really do not avoid each other. | Use `SMOOTH_ANIMATION = true` (slower) to see real avoidance. |

**A finished simulation does not prove that no collisions happened.** The avoidance modules reduce conflicts but do
not guarantee collision-free motion, so check the safety record of the run (smallest gap, overlap steps, swaps).
It covers free robots and formed transport teams, not parts lying on the ground.

## 9. Licences

**Everything needed for the default experiments is free.** The heuristic rules and the metaheuristics need no
solver at all (they are part of the repository's Julia code); the MILP
methods work with **HiGHS** (MIT licence) or **GLPK** (GPL-3.0), both installed automatically with the project.

| Component | Licence | Notes |
|---|---|---|
| ConstructionBots.jl | MIT (`LICENSE`, © 2021 Kyle Brown and contributors) | Free to use, modify and share with the copyright notice. Please cite the paper (see `README.md`). |
| This notebook and its helper files | Part of the repository; same MIT terms as the repository unless your lab decides otherwise. | |
| HiGHS solver | MIT | Free, including commercial use. |
| GLPK solver (and GLPK.jl) | GNU GPL-3.0 | Free to use; redistributing software that *includes* GLPK has GPL obligations. |
| ECOS (used internally for the site layout) | GPL-3.0 (ECOS.jl wrapper: MIT) | Installed automatically. |
| JuMP | MPL-2.0 | |
| RVO2 library (sisl/Python-RVO2 fork) | Apache-2.0 (© University of North Carolina at Chapel Hill) | Installed separately (`SETUP.md`). |
| LDraw Parts Library | CC BY 2.0 / CC BY 4.0 (see `CAreadme.txt` in the library) | Download it yourself from ldraw.org; do not commit it to the repository. If you share images or parts, give attribution to LDraw.org. LEGO® is a trademark of the LEGO Group, which does not endorse this project. |
| **Gurobi** solver | **Commercial** (Gurobi EULA) | **Optional.** The `Gurobi` Julia package installs the solver program, but it will not run without a **licence of your own**. |

**About Gurobi licences.** Free academic licences are offered to students, faculty and staff of accredited
degree-granting institutions for non-commercial teaching and research. A *Named-User* academic licence is
personal and tied to one computer: having one does **not** give other lab members the right to use Gurobi, and
licence files must never be put in this notebook or in the repository. Each user should check their own
eligibility and obtain their own licence (see `SETUP.md`, "Optional: Gurobi", and https://www.gurobi.com/academia/).
Always confirm the current terms on Gurobi's website; they can change.

This notebook uses **no web services or AI APIs**; every experiment runs locally.

## 10. Known limitations

* **RVO2 is always required.** Even with RVO2 switched off, the simulator creates an RVO2 simulation object.
* **Dispersion only acts together with RVO2.** In this version of the code, the dispersion (potential-field)
  velocity adjustment is applied inside the RVO2 branch of the controller. With RVO2 off, the dispersion setting
  has little or no effect on how robots move.
* **Results older than this version are not comparable.** Results used to change whenever the package was recompiled.
  That is fixed, so numbers from before the fix (with no `source_hash`) cannot be compared with current ones.
  A change to `src/` or to a package version can still change results on purpose: `source_hash` shows when it did,
  and earlier results are not reused then. Results on another operating system were not tested and may differ slightly.
* **Metaheuristics search the task order only, and they minimize the plan.** Robot pairing and the build order follow
  Greedy's rules, so they explore a part of all possible schedules. They minimize the *planned* completion time. The
  simulated one includes traffic, can rank the algorithms differently, and is what counts (see Section 6). A plain search
  can be much slower than Greedy in simulation and can even get stuck, as it did on Colored 8×8. `SIMULATE_TOP_PLANS`
  prevents a result worse than Greedy's, at the cost of extra simulations.
* **The simulator abstracts physics.** Velocity changes are instant unless you pass `max_acceleration` to
  `run_lego_demo`, robots can swap positions instantly, the top speed is 4 m/s whatever the model's scale, and parts are
  loaded faster than robots drive on small models. So completion times are not comparable across models and do not
  predict real-robot times. No part of this has been calibrated against real hardware.
* **Experiments vary only the robots' start positions** (and the metaheuristics' random numbers) by default. The parts'
  locations are fixed unless you use `scripts/method_comparison.jl --placements`.
* **MILP results depend on the time limit.** With the default limits the MILP solver usually stops at the time
  limit without proving its answer optimal; its schedule can be *worse* than Greedy's. Report the time limit with
  any MILP result.
* **Computer-time measurements** vary between computers and include compilation on the first run of each kind
  after a restart (the `includes_one_time_compilation` column marks these runs).
* **Determinism.** Runs of the rules and the metaheuristics with the same code, settings and seed give the same
  result, also after a recompile (a metaheuristic only if it finishes its budget before the time limit). The seed
  controls the robots' starting positions. **MILP runs that stop at the time limit are not exactly reproducible**: how far the solver gets in
  the allowed time depends on the computer's speed and load. On the test computer, three identical
  MILP + HiGHS runs (Tractor, seed 1, 30 s) gave completion times of 55.9, 50.3 and 49.8 s. Use several seeds
  and repetitions, and report the time limit.
* **Saving an animation** starts MeshCat's viewer server on `127.0.0.1` (this computer only) during the run; it
  is closed when the run ends. The saved HTML file contains the viewer program and all scene data and works
  without any server or internet connection. `show_animation` embeds it in the cell output (files up to
  30 MB), so it shows in any notebook front end and survives a kernel restart; larger files are served
  read-only from the notebook's Julia session (`127.0.0.1`).
* **Safety is recorded, not enforced.** Brief overlaps can occur even with all avoidance methods on, and the completion
  time does not penalize them. Report the safety record with any completion time.
* Large models (hundreds to thousands of parts) can take from minutes to hours and much memory.